In [0]:
FEATURE_TABLE_NAME = "workspace.marketing_campaign.gold_customer_features"

features_df = spark.table(FEATURE_TABLE_NAME)

print(f"Feature table: {FEATURE_TABLE_NAME}")
print(f"Rows: {features_df.count()}")
print(f"Columns: {len(features_df.columns)}")

In [0]:
model_df = features_df.select(
    "education",
    "marital_status",
    "income",
    "customer_age",
    "customer_tenure_days",
    "has_children",
    "recency",
    "total_spend",
    "total_purchases",
    "numwebvisitsmonth",
    "acceptedcmp1",
    "acceptedcmp2",
    "acceptedcmp3",
    "acceptedcmp4",
    "acceptedcmp5",
    "accepted_previous_campaign",
    "complain",
    "response"
).dropna()

train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_df.count()}")
print(f"Test rows: {test_df.count()}")

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

categorical_columns = ["education", "marital_status"]

numeric_columns = [
    "income",
    "customer_age",
    "customer_tenure_days",
    "has_children",
    "recency",
    "total_spend",
    "total_purchases",
    "numwebvisitsmonth",
    "acceptedcmp1",
    "acceptedcmp2",
    "acceptedcmp3",
    "acceptedcmp4",
    "acceptedcmp5",
    "accepted_previous_campaign",
    "complain"
]

indexers = [
    StringIndexer(
        inputCol=column_name,
        outputCol=f"{column_name}_index",
        handleInvalid="keep"
    )
    for column_name in categorical_columns
]

encoders = [
    OneHotEncoder(
        inputCol=f"{column_name}_index",
        outputCol=f"{column_name}_encoded"
    )
    for column_name in categorical_columns
]

assembler = VectorAssembler(
    inputCols=numeric_columns + [f"{column_name}_encoded" for column_name in categorical_columns],
    outputCol="features"
)

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="response",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=20
)

pipeline = Pipeline(
    stages=indexers + encoders + [assembler, logistic_regression]
)

model = pipeline.fit(train_df)
predictions_df = model.transform(test_df)

print("Model retrained for evaluation.")

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="response",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="response",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="response",
    predictionCol="prediction",
    metricName="f1"
)

roc_auc = auc_evaluator.evaluate(predictions_df)
accuracy = accuracy_evaluator.evaluate(predictions_df)
f1_score = f1_evaluator.evaluate(predictions_df)

print(f"ROC AUC: {roc_auc}")
print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1_score}")

In [0]:
display(
    predictions_df
    .groupBy("response", "prediction")
    .count()
    .orderBy("response", "prediction")
)

In [0]:
display(
    predictions_df
    .groupBy("response")
    .count()
    .orderBy("response")
)

In [0]:
display(
    predictions_df
    .groupBy("prediction")
    .count()
    .orderBy("prediction")
)

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def get_response_probability(probability_vector):
    return float(probability_vector[1])

get_response_probability_udf = udf(get_response_probability, DoubleType())

predictions_with_probability_df = predictions_df.withColumn(
    "response_probability",
    get_response_probability_udf("probability")
)

In [0]:
display(
    predictions_with_probability_df.select(
        "education",
        "marital_status",
        "income",
        "customer_age",
        "total_spend",
        "total_purchases",
        "response",
        "prediction",
        "response_probability"
    ).orderBy("response_probability", ascending=False)
)

In [0]:
print("Model evaluation notes:")
print("- ROC AUC measures how well the model separates responders from non-responders.")
print("- Accuracy measures the share of correct predictions.")
print("- F1 score balances precision and recall.")
print("- If response=1 is rare, F1 and ROC AUC are usually more useful than accuracy alone.")
print("- Customers with higher response_probability are better campaign targeting candidates.")